In [1]:
import torch
import requests
from PIL import Image
from transformers import AutoProcessor, AutoModel

In [ ]:
import torch
import requests
from PIL import Image
# ZMIANA 1: Importujemy AutoImageProcessor i AutoTokenizer osobno
from transformers import AutoImageProcessor, AutoTokenizer, AutoModel

# Ustawienia urządzenia
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "facebook/metaclip-2-worldwide-giant-378"

print(f"Ładowanie procesorów i modelu na: {device}...")

# ZMIANA 2: Ładujemy komponenty oddzielnie
image_processor = AutoImageProcessor.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModel.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16, 
    attn_implementation="sdpa"
).to(device)

# Przygotowanie danych wejściowych
url = "http://images.cocodataset.org/val2017/000000039769.jpg" # Zdjęcie kotów na kanapie
image = Image.open(requests.get(url, stream=True).raw)
labels = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

# ZMIANA 3: Przetwarzamy tekst i obraz niezależnie od siebie
image_inputs = image_processor(images=image, return_tensors="pt").to(device)
text_inputs = tokenizer(text=labels, padding=True, return_tensors="pt").to(device)

# BAZOWY PRZEBIEG (bez modyfikacji)
with torch.no_grad():
    # ZMIANA 4: Podajemy argumenty wprost do modelu
    outputs_base = model(
        input_ids=text_inputs.input_ids,
        attention_mask=text_inputs.attention_mask,
        pixel_values=image_inputs.pixel_values
    )
    
probs_base = outputs_base.logits_per_image.softmax(dim=1)[0]
print("\n--- WYNIK BAZOWY (BEZ INGERENCJI) ---")
for label, prob in zip(labels, probs_base):
    print(f"{label}: {prob.item():.4f}")

Ładowanie procesorów i modelu na: cpu...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

C:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--facebook--metaclip-2-worldwide-giant-378. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# 1. Definiujemy funkcję haka (hook), która zmodyfikuje aktywacje
def modify_activation_hook(module, inputs, output):
    """
    module: informacja o warstwie, do której podpięty jest hak
    inputs: to, co weszło do warstwy
    output: to, co z niej wyszło (to chcemy zmodyfikować!)
    """
    # W modelach z HF warstwy często zwracają krotkę (tuple), gdzie pierwszy element to 'hidden_states'
    is_tuple = isinstance(output, tuple)
    hidden_states = output[0] if is_tuple else output
    
    print(f"[HOOK] Przechwycono aktywacje o kształcie: {hidden_states.shape}")
    
    # --- TUTAJ WPROWADZASZ SWOJĄ MODYFIKACJĘ ---
    # Przykład: Zerujemy (wygaszamy) wszystkie sygnały w tej warstwie
    # Możesz tu dodać szum, pomnożyć przez skalar, albo podmienić na wektor z innego obrazka
    modified_hidden_states = hidden_states * 0.0 
    
    # Zwracamy zmodyfikowany output z powrotem do modelu
    if is_tuple:
        return (modified_hidden_states,) + output[1:]
    else:
        return modified_hidden_states

# 2. Wybieramy warstwę, którą chcemy zhakować
# Modele CLIP składają się z vision_model i text_model. 
# Wejdźmy w głąb wizji (ViT-bigG ma kilkadziesiąt warstw, wybieramy np. 20-tą)
layer_to_hack = model.vision_model.encoder.layers[20]

# 3. Rejestrujemy hak (przypinamy go do warstwy)
hook_handle = layer_to_hack.register_forward_hook(modify_activation_hook)

print("\n--- URUCHAMIAMY MODEL Z MODYFIKACJĄ ---")
# 4. Ponownie przepuszczamy dane przez model
with torch.no_grad():
    outputs_modified = model(**inputs)

# 5. Odpinamy hak (BARDZO WAŻNE, żeby nie popsuć modelu na przyszłość)
hook_handle.remove()

# Sprawdzamy wyniki po naszej modyfikacji
probs_modified = outputs_modified.logits_per_image.softmax(dim=1)[0]
print("\n--- WYNIK PO ZEROWANIU WARSTWY 20 ---")
for label, prob in zip(labels, probs_modified):
    print(f"{label}: {prob.item():.4f}")